# Análisis de errores

### Importación de librerías

In [8]:
import pandas as pd
import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
import sys
from sklearn.model_selection import GroupShuffleSplit
from functools import partial

sys.path.insert(0, str(Path.cwd().parent))
from scripts.funcion_ex import ex
from scripts.consulta_sql import ejecutar_consulta
from scripts.inferencia import generar_sql_inferencia

### Rutas

In [2]:
ruta_datos = '../datos/validacion.xlsx'
esquemas = '../esquemas/'
ruta_adapter_r8 = Path.cwd().parent / 'modelos' / 'modelo_r8'
ruta_adapter_r16 = Path.cwd().parent / 'modelos' / 'modelo_r16'
ROOT = Path.cwd()
SEMILLA = 2026

### Importación de datos

In [3]:
datos = pd.read_excel(ruta_datos)
datos.head(2)

,nivel,ponderacion,id_esquema,prompt,sql
0,Fácil,1,sales_clothes,Mostrar todas las columnas de las ventas regis...,SELECT * FROM Sales;
1,Fácil,1,sales_clothes,Muestra el identificador y nombre de todos los...,"SELECT id_costumer, name_costumer FROM Costume..."


### Agregando el esquema a los datos

In [4]:
# Función para leer el esquema.
def obtener_esquema(id_esquema):
    ruta = esquemas +  f'{id_esquema}.sql'

    with open(ruta, 'r', encoding='utf-8') as archivo:
        return archivo.read()

datos['esquema'] = datos['id_esquema'].apply(obtener_esquema)
datos.head(2)

,nivel,ponderacion,id_esquema,prompt,sql,esquema
0,Fácil,1,sales_clothes,Mostrar todas las columnas de las ventas regis...,SELECT * FROM Sales;,"CREATE TABLE `Sales` (`id_sale` DECIMAL(18,2),..."
1,Fácil,1,sales_clothes,Muestra el identificador y nombre de todos los...,"SELECT id_costumer, name_costumer FROM Costume...","CREATE TABLE `Sales` (`id_sale` DECIMAL(18,2),..."


## Modelo Base

### Cargas de la configuración

In [5]:
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Cargar configuración.
with open(ROOT / 'config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

# Nombre del modelo
nombre_modelo = config['modelo_base']

# Semilla de aleatoriedad
semilla = config.get('semilla', SEMILLA)
torch.manual_seed(semilla)

# Dispositivo y precisión.
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if dispositivo == 'cuda' else torch.float32

### Carga del tokenizer y del modelo

In [6]:
# Tokenizer.
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# Modelo sin Adapter.
modelo = AutoModelForCausalLM.from_pretrained(nombre_modelo, torch_dtype=dtype).to(dispositivo)
modelo.generation_config.do_sample = False
modelo.generation_config.temperature = 1.0
modelo.generation_config.top_p = 1.0
modelo.generation_config.top_k = 50

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


### Evaluación de error

In [ ]:
ex_valor, _ = ex(modelo=modelo, tokenizer=tokenizer, datos=datos)

print(f"EX: {ex_valor:.4f}")
print(f"Error: {1- ex_valor:.4f}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


EX: 0.2208
Error: 0.7792


### Liberando memoria

In [8]:
del modelo
del tokenizer

## Modelo con Adapter (r=8)

### Modelo

In [5]:
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Cargar configuración.
with open(ROOT / 'config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

# Nombre del modelo
nombre_modelo = config['modelo_base']

# Semilla de aleatoriedad
semilla = config.get('semilla', SEMILLA)
torch.manual_seed(semilla)

# Dispositivo y precisión.
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if dispositivo == 'cuda' else torch.float32

# Tokenizer.
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# Modelo sin Adapter.
modelo = AutoModelForCausalLM.from_pretrained(nombre_modelo, torch_dtype=dtype).to(dispositivo)
modelo.generation_config.do_sample = False
modelo.generation_config.temperature = 1.0
modelo.generation_config.top_p = 1.0
modelo.generation_config.top_k = 50

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


### Adapter

In [6]:
#  Configuración del adapter.
config_lora = PeftConfig.from_pretrained(ruta_adapter_r8)
modelo = PeftModel.from_pretrained(modelo, str(ruta_adapter_r8), is_trainable=False)

### Evaluación de error

In [ ]:
ex_valor, _ = ex(modelo=modelo, tokenizer=tokenizer, datos=datos)

print(f"EX: {ex_valor:.4f}")
print(f"Error: {1- ex_valor:.4f}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


EX: 0.2458
Error: 0.7542


### Liberando memoria

In [8]:
del modelo
del tokenizer

## Modelo con Adapter (r=16)

### Modelo

In [5]:
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Cargar configuración.
with open(ROOT / 'config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

# Nombre del modelo
nombre_modelo = config['modelo_base']

# Semilla de aleatoriedad
semilla = config.get('semilla', SEMILLA)
torch.manual_seed(semilla)

# Dispositivo y precisión.
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if dispositivo == 'cuda' else torch.float32

# Tokenizer.
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# Modelo sin Adapter.
modelo = AutoModelForCausalLM.from_pretrained(nombre_modelo, torch_dtype=dtype).to(dispositivo)
modelo.generation_config.do_sample = False
modelo.generation_config.temperature = 1.0
modelo.generation_config.top_p = 1.0
modelo.generation_config.top_k = 50

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


### Adapter

In [6]:
#  Configuración del adapter.
config_lora = PeftConfig.from_pretrained(ruta_adapter_r16)
modelo = PeftModel.from_pretrained(modelo, str(ruta_adapter_r16), is_trainable=False)

### Evaluación de error

In [ ]:
ex_valor, _ = ex(modelo=modelo, tokenizer=tokenizer, datos=datos)

print(f"EX: {ex_valor:.4f}")
print(f"Error: {1- ex_valor:.4f}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


EX: 0.2875
Error: 0.7125


### Liberando memoria

In [8]:
del modelo
del tokenizer

## Tipos de errores

### Modelo

In [5]:
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Cargar configuración.
with open(ROOT / 'config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

# Nombre del modelo
nombre_modelo = config['modelo_base']

# Semilla de aleatoriedad
semilla = config.get('semilla', SEMILLA)
torch.manual_seed(semilla)

# Dispositivo y precisión.
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if dispositivo == 'cuda' else torch.float32

# Tokenizer.
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# Modelo sin Adapter.
modelo = AutoModelForCausalLM.from_pretrained(nombre_modelo, torch_dtype=dtype).to(dispositivo)
modelo.generation_config.do_sample = False
modelo.generation_config.temperature = 1.0
modelo.generation_config.top_p = 1.0
modelo.generation_config.top_k = 50

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


### Mejor Adapter

In [6]:
#  Configuración del adapter.
config_lora = PeftConfig.from_pretrained(ruta_adapter_r16)
modelo = PeftModel.from_pretrained(modelo, str(ruta_adapter_r16), is_trainable=False)

### Errores

In [9]:
ex_valor, resultados = ex(modelo=modelo, tokenizer=tokenizer, datos=datos)

### Resultados de errores

In [10]:
resultados.head()

,indice,nombre_db,prompt,sql_real,sql_predicho,ponderacion,acierto,categoria,mensaje_error
0,0,sales_clothes,Mostrar todas las columnas de las ventas regis...,SELECT * FROM Sales;,SELECT * FROM Sales,1,1,None,None
1,1,sales_clothes,Muestra el identificador y nombre de todos los...,"SELECT id_costumer, name_costumer FROM Costume...","SELECT id_costumer, name_costumer FROM Costum...",1,1,None,None
2,2,sales_clothes,"Obtén las prendas, sus identificadores y preci...","SELECT id_clothe, name_clothe, price FROM Clot...","SELECT name_clothe, id_clothe, price FROM Cl...",1,1,None,None
3,3,sales_clothes,"Dame las prendas, sus identificares y tallas d...","SELECT id_clothe, name_clothe, size FROM Cloth...","SELECT name_clothe, id_clothe, size FROM Clo...",1,1,None,None
4,4,sales_clothes,"Identifica las prendas con sus precios, cuyo p...","SELECT name_clothe, price FROM Clothes WHERE p...","SELECT name_clothe, price FROM Clothes WHERE ...",1,1,None,None


In [11]:
resultados['categoria'].value_counts()

categoria
Resultado distinto             35
Columna o tabla inexistente    33
Sintaxis                        1
Agregación                      1
Name: count, dtype: int64

### Almacenando los resultados

In [12]:
resultados.to_csv('../resultados/validacion_propia.csv', index=False)

## Inferencia

### Few-shot
Se realizará este experimento con el fin de que pueda tener a la mano algunos ejemplos a los cuales debe acoplarse. Esta estrategia podrá ayudarnos con los distintos errores que tengamos.

### Ejemplos

In [13]:
# Rutas del proyecto.
ROOT = Path.cwd().parent

# Cargar datos originales.
datos_entrenamiento = pd.read_csv(ROOT / 'datos' / 'entrenamiento.csv')
datos_entrenamiento = datos_entrenamiento.drop(columns=['valida'])

# División de los datos
division = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEMILLA)
idx_train, idx_test = next(division.split(datos_entrenamiento, groups=datos_entrenamiento['db_id']))
df_test = datos_entrenamiento.iloc[idx_test].copy()

# Esquemas
df_test['esquema'] = df_test['db_id'].apply(obtener_esquema)

# Seleccionar cinco ejemplos reproducibles.
ejemplos = (df_test.sample(n=15, random_state=SEMILLA)[['esquema', 'pregunta', 'sql']].to_dict(orient='records'))

### Experimento

In [14]:
# Incorporar los ejemplos a la función few-shot.
inferencia_few_shot = partial(generar_sql_inferencia, ejemplos=ejemplos)

# Evaluar todas las preguntas.
ex_few_shot, resultados_few_shot = ex(modelo=modelo,tokenizer=tokenizer, datos=datos,
                                      max_new_tokens=1500, funcion_inferencia=inferencia_few_shot)

print(f"EX few-shot: {ex_few_shot:.4f}")

EX few-shot: 0.2125


### Conclusión

El few-shot no mejoró el modelo, incluso lo empeoró e hizo que fuera más lento. Se dejará el modelo tal cual está.

### Liberando memoria

In [15]:
del modelo
del tokenizer